# Synthetic Probe Test — Per-Judge, Per-Train-Dataset Evaluation

For each `(judge_model, train_dataset)` combination, evaluate the trained probe
across four test settings and produce publication-ready figures and a summary table.

**Four test settings** (color = test dataset; linestyle = real vs. synthetic):
1. **Syn {train\_dataset} (within)** — synthetic held-out test split, same domain as training
2. **Syn {other\_dataset} (cross)** — synthetic held-out test split, cross-domain
3. **Real {train\_dataset} (within)** — real extraction outputs, same domain as training
4. **Real {other\_dataset} (cross)** — real extraction outputs, cross-domain

**Per-combination outputs**:
- **Calibration curves**: probe (solid) vs. NTP baseline (faint dotted), one line per test setting
- **Threshold sweep** (2 × 2 grid): recovery vs. hallucination rate as decision threshold varies
- **Summary table**: one row per (test setting × method) at threshold = 0.5

**Frontier judge baselines** evaluated on synthetic test data are independent of the
probe analysis and appear in their own section.

**Label definition** (`LABEL_MODE = 'union'`): a real extraction is positive if it matches
a ground-truth row **or** the majority-vote frontier judge says valid.

**Prerequisites**: Run `synthetic_probe_analysis.ipynb` for each `(dataset, judge_model)`
combination before running this notebook.

In [ ]:
import sys
from pathlib import Path
%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

from analysis.loaders import (
    load_activations, load_combined_judgements,
    load_extraction, load_ground_truth, load_trained_probe,
    cached_match, load_synthetic_activations, load_synthetic_layer_outputs, 
    load_synthetic_responses,
)
from scholarlm.utils.calibration import reliability_diagram_data, rescale_probabilities_em
from scholarlm.utils.unit_conversion import apply_unit_conversion
from experiments.run_extraction import load_dataset_config
import paths

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "text.usetex": False,
    "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "legend.title_fontsize": 9,
    "axes.linewidth": 0.6,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.major.size": 3, "ytick.major.size": 3,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "lines.linewidth": 1.2, "lines.markersize": 4,
    "legend.frameon": False,
    "figure.dpi": 150, "savefig.dpi": 300,
    "savefig.format": "pdf", "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

FIGURES_DIR = "../figures/synthetic_probe/"
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
DATASETS = ['pond', 'nfix']
EXTRACTION_MODEL = 'gemma-3-27b'
JUDGE_MODELS = ['llama-3.1-8b', 'mistral-7b', 'qwen-2.5-7b']

# Extraction date per test dataset
EXTRACTION_DATES = {
    'pond': '2026_04_30',
    'nfix': '2026_04_30',
}

# Judge date for synthetic test activations: {dataset: {judge_model: date_str | None}}
# None → auto-detect latest
JUDGE_DATES_SYN = {
    'pond': {
        'llama-3.1-8b': '2026_05_04',
        'mistral-7b': '2026_05_04',
        'qwen-2.5-7b': '2026_05_04',
    },
    'nfix': {
        'llama-3.1-8b': '2026_05_04',
        'mistral-7b': '2026_05_04',
        'qwen-2.5-7b': '2026_05_04',
    },
}

# Judge date for real activations: {test_dataset: {judge_model: date_str | None}}
# None → auto-detect latest
JUDGE_DATES_REAL = {
    'pond': {
        'llama-3.1-8b': '2026_05_04',
        'mistral-7b': '2026_05_04',
        'qwen-2.5-7b': '2026_05_04',
    },
    'nfix': {
        'llama-3.1-8b': '2026_05_04',
        'mistral-7b': '2026_05_04',
        'qwen-2.5-7b': '2026_05_04',
    },
}

THRESHOLD_SWEEP = np.linspace(0.5, 0.95, 15)  # thresholds for operating-curve plot
EDGE_THRESHOLD  = 1 / 3  # minimum fuzzy weight to count as a match

# Visual style
JUDGE_COLORS = {
    'llama-3.1-8b': '#3984e0',
    'mistral-7b':   '#2eb07b',
    'qwen-2.5-7b':   '#d946a6',
}

## Load Trained Probes

In [ ]:
# probe_cache[train_dataset][judge_model] = probe_data dict
probe_cache = {}
for ds in DATASETS:
    probe_cache[ds] = {}
    for jm in JUDGE_MODELS:
        probe_cache[ds][jm] = load_trained_probe(ds, jm)
        top = probe_cache[ds][jm]['top_k_heads']
        print(f'  {ds} / {jm}: top heads={top}  pi_tr={probe_cache[ds][jm]["train_prevalence"]:.3f}')

## Analysis Helpers

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import brier_score_loss


def _probe_metrics(probs, y_true, threshold=0.5):
    """Compute metrics at a fixed threshold. Returns dict."""
    probs   = np.asarray(probs)
    y_true  = np.asarray(y_true, dtype=bool)
    preds   = probs > threshold
    tp  = int(( preds &  y_true).sum())
    tn  = int((~preds & ~y_true).sum())
    fp  = int(( preds & ~y_true).sum())
    fn  = int((~preds &  y_true).sum())
    n   = len(y_true)
    acc   = (tp + tn) / n
    prec  = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    rec   = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    f1    = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else float('nan')
    auroc = roc_auc_score(y_true, probs) if y_true.sum() > 0 and (~y_true).sum() > 0 else float('nan')
    ece   = reliability_diagram_data(probs, y_true)['ece']
    bs    = float(brier_score_loss(y_true, probs))
    p_pos = float(y_true.mean())
    bss   = 1.0 - bs / (p_pos * (1 - p_pos)) if p_pos not in (0.0, 1.0) else float('nan')
    return dict(acc=acc, prec=prec, rec=rec, f1=f1, auroc=auroc,
                ece=ece, bs=bs, bss=bss, n=n)


def compute_rates(probs, threshold, real_test_labels, test_edges, n_gt):
    """Recovery rate and hallucination rate at a given threshold.

    At threshold t:
      accepted      = probs > t (accepted extractions)
      recovery      = fraction of GT rows matched by at least one accepted extraction
      hallucination = fraction of accepted extractions that are invalid
                      (not GT-matched and not judged valid by frontier judge)

    Parameters
    ----------
    probs            : 1-D array aligned with the test set
    threshold        : float
    real_test_labels : bool array aligned with the test set (union labels)
    test_edges       : list of (gt_idx, test_pos) pairs (pre-filtered to test set)
    n_gt             : total number of GT rows
    """
    accepted = np.asarray(probs) > threshold
    n_accepted = accepted.sum()
    if n_accepted == 0:
        return float('nan'), float('nan')
    gt_covered = np.zeros(n_gt, dtype=bool)
    for gt_idx, test_pos in test_edges:
        if accepted[test_pos]:
            gt_covered[gt_idx] = True
    recovery      = float(gt_covered.mean())
    hallucination = float(1 - real_test_labels[accepted].mean())
    return recovery, hallucination


def compute_rates_syn(probs, threshold, labels):
    """Recovery (recall on positives) and hallucination (FP rate among accepted) at threshold.

    Simpler than compute_rates: labels are direct binary labels, no GT matching needed.
    """
    probs  = np.asarray(probs)
    labels = np.asarray(labels, dtype=bool)
    accepted   = probs > threshold
    n_accepted = int(accepted.sum())
    if n_accepted == 0:
        return float('nan'), float('nan')
    n_pos = int(labels.sum())
    recovery      = float((accepted & labels).sum() / n_pos) if n_pos > 0 else float('nan')
    hallucination = float((accepted & ~labels).sum() / n_accepted)
    return recovery, hallucination

## Matching Configuration

In [ ]:
_SUPER_MAP  = str.maketrans('⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺', '0123456789-+')
_SUB_MAP    = str.maketrans('₀₁₂₃₄₅₆₇₈₉₋₊', '0123456789-+')
_SCRIPT_MAP = {**_SUPER_MAP, **_SUB_MAP}

_LATEX_RE    = re.compile(r'[\^_]\{([^}]*)\}|[\^_]([+-]?\d+)')
_COMPOUND_RE = re.compile(r'(\w+)[\s\-]([A-Z][a-zA-Z0-9]*)')


def nfix_clean_unit(s: str) -> str:
    if not isinstance(s, str):
        return s
    s = s.translate(_SCRIPT_MAP)
    s = _LATEX_RE.sub(lambda m: m.group(1) if m.group(1) is not None else m.group(2), s)
    s = s.replace('µ', 'u').replace('μ', 'u')
    s = _COMPOUND_RE.sub(r'\1-\2', s)
    s = re.sub(r'\byr\b', 'y', s)
    s = s.lower()
    s = re.sub(r'\bday\b', 'd', s)
    s = re.sub(r'\bhr\b',  'h', s)
    return s


def get_matching_config(dataset):
    if dataset == 'pond':
        strict = {'document_id': 'document_id', 'attribute': 'attribute',
                  'value': 'converted_value', 'units': 'units'}
        fuzzy  = {'name': 'name', 'location': 'location', 'ecosystem': 'ecosystem'}
    elif dataset == 'nfix':
        strict = {'document_id': 'document_id', 'attribute': 'attribute',
                  'value': 'converted_value', 'units': 'units'}
        fuzzy  = {'name': 'name', 'site_type': 'site_type'}
    else:
        raise ValueError(f'Unknown dataset: {dataset}')
    return strict, fuzzy

## Load and Cache Real Test Data

For each test dataset: load extraction results, combined judgements, and ground truth;
compute union labels (GT-matched **or** frontier judge says valid); run `cached_match`.

In [ ]:
test_data = {}  # test_data[dataset] = dict with pre-loaded data

for ds in DATASETS:
    print(f'Loading test data for {ds}...')
    config  = load_dataset_config(ds)
    records = load_extraction(ds, EXTRACTION_MODEL, EXTRACTION_DATES[ds])
    ext_df  = pd.DataFrame(records)
    ext_df  = apply_unit_conversion(ext_df, config.unit_conversion_table)

    if ds == 'nfix':
        ext_df['attribute'] = ext_df['attribute'].map({
            'nfix_rate_areal': 'nfix_rate', 'nfix_rate_volumetric': 'nfix_rate',
            'nfix_rate_mass':  'nfix_rate', 'nfix_rate': 'nfix_rate',
        })
        ext_df['units'] = ext_df['units'].apply(nfix_clean_unit)

    real_df = pd.DataFrame(load_combined_judgements(ds, EXTRACTION_MODEL, EXTRACTION_DATES[ds]))
    gt_df   = load_ground_truth(config)

    strict, fuzzy = get_matching_config(ds)
    cache_path = paths.extraction(ds, EXTRACTION_MODEL, EXTRACTION_DATES[ds]) / 'match_cache.pkl'
    matching, edges, edge_weights = cached_match(
        gt_df, ext_df,
        strict_matching=strict,
        fuzzy_matching=fuzzy,
        fuzzy_threshold=0.0,
        cache_path=cache_path,
    )

    ex_edge_exists = np.zeros(len(ext_df), dtype=bool)
    for (gt_idx, ex_idx), w in zip(edges, edge_weights):
        if w > EDGE_THRESHOLD:
            ex_edge_exists[int(ex_idx)] = True
    jlabels     = real_df['judgement_combined'].to_numpy(dtype=bool)
    real_labels = jlabels | ex_edge_exists

    test_data[ds] = {
        'extraction_df':   ext_df,
        'real_df':         real_df,
        'ground_truth_df': gt_df,
        'real_labels':     real_labels,
        'ex_edge_exists':  ex_edge_exists,
        'edges':           edges,
        'edge_weights':    edge_weights,
    }

    pos = real_labels.sum()
    print(f'  {ds}: {len(ext_df)} extractions, {len(gt_df)} GT rows, '
          f'pos={pos} ({pos / len(real_labels):.1%})')
    print()

## Frontier Judge Baselines on Synthetic Test Data

Evaluate frontier / alternative judges on the held-out synthetic test split.
These backends return binary verdicts, so we report accuracy, precision, recall, and F1
(AUROC is omitted).  Since real-data labels are partly constructed by LLM judges,
this section validates how reliable that labelling process is.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
ALT_JUDGE_MODELS = [
    ('gpt-4o-mini',               None),
    ('claude-haiku-4-5-20251001', None),
]

# ── Evaluate each alternative judge on each dataset's synthetic test split ────
for ds in DATASETS:
    print(f'\nJudge baselines — {ds} synthetic test set:')

    # Load ground-truth labels from the first local judge model (same for all judges)
    ref_jm    = JUDGE_MODELS[0]
    ref_jdate = JUDGE_DATES_SYN[ds][ref_jm]
    ref_resp  = load_synthetic_responses(ds, ref_jm, ref_jdate, split='test')
    ref_df    = pd.DataFrame(ref_resp)
    gt_by_id  = dict(zip(ref_df['measurement_id'],
                         (ref_df['label'] == 'valid').tolist()))

    rows = []
    for model_name, judge_date in ALT_JUDGE_MODELS:
        try:
            responses = load_synthetic_responses(ds, model_name, judge_date, split='test')
        except FileNotFoundError as e:
            print(f'  [{model_name}] not found — skipping\n    {e}')
            continue

        y_true, y_pred, n_null = [], [], 0
        for r in responses:
            j = r.get('judgement')
            if j is None:
                n_null += 1
                continue
            mid = r['measurement_id']
            if mid not in gt_by_id:
                continue
            y_true.append(gt_by_id[mid])
            y_pred.append(bool(j))

        y_true = np.array(y_true, dtype=bool)
        y_pred = np.array(y_pred, dtype=bool)
        n      = len(y_true)
        acc    = float((y_pred == y_true).mean()) if n > 0 else float('nan')
        tp = int((y_pred &  y_true).sum())
        fp = int((y_pred & ~y_true).sum())
        fn = int((~y_pred & y_true).sum())
        p     = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
        r_val = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
        f1    = 2 * p * r_val / (p + r_val) if (p + r_val) > 0 else float('nan')
        rows.append(dict(model=model_name, n=n, n_null=n_null,
                         acc=acc, prec=p, rec=r_val, f1=f1))

    if rows:
        alt_df = pd.DataFrame(rows).set_index('model')
        display(alt_df.style.format(
            {'acc': '{:.3f}', 'prec': '{:.3f}', 'rec': '{:.3f}', 'f1': '{:.3f}'}
        ))

## Probe Analysis — Per (Judge Model, Train Dataset)

For each `(judge_model, train_dataset)` combination the probe is evaluated across
all four test settings.

**Visual encoding**
- Color = test dataset: `pond` (blue `#2166ac`) · `nfix` (red `#d6604d`)
- Linestyle = data type: real (solid `—`) · synthetic (dashed `- -`)
- NTP baseline = same color, faint dotted (α = 0.30)

In [ ]:
# Visual encoding constants
_DS_COLORS = {'pond': '#2166ac', 'nfix': '#d6604d'}
_DTYPE_LS  = {'real': '-', 'syn': '--'}


def analyze_probe(judge_model, train_dataset):
    """Calibration curves, threshold sweep, and metrics table for one
    (judge_model, train_dataset) pair evaluated across four test settings."""
    other_ds = [ds for ds in DATASETS if ds != train_dataset][0]
    pd_data  = probe_cache[train_dataset][judge_model]
    top_k    = pd_data['top_k_heads']
    header   = f'{judge_model}  |  trained on {train_dataset}'
    SEP      = '=' * 72
    print(f'\n{SEP}\n  {header}\n{SEP}')

    # ── Collect data for each test setting ────────────────────────────────
    setting_results = []

    for dtype in ('syn', 'real'):
        for test_ds in (train_dataset, other_ds):
            domain   = 'within' if test_ds == train_dataset else 'cross'
            dtype_str = 'Syn' if dtype == 'syn' else 'Real'
            s_label  = f'{dtype_str} {test_ds} ({domain})'
            color    = _DS_COLORS[test_ds]
            ls       = _DTYPE_LS[dtype]

            if dtype == 'syn':
                jdate    = JUDGE_DATES_SYN[test_ds][judge_model]
                syn_act  = load_synthetic_activations(test_ds, judge_model, jdate, split='test')
                syn_resp = load_synthetic_responses(test_ds, judge_model, jdate, split='test')
                syn_df_s = pd.DataFrame(syn_resp)
                mids     = syn_df_s['measurement_id'].tolist()
                labels   = (syn_df_s['label'] == 'valid').to_numpy(dtype=bool)

                X = np.concatenate([
                    np.stack([
                        np.array(syn_act[str(mid)], dtype=np.float32)[l, h, :]
                        for mid in mids
                    ], axis=0)
                    for l, h in top_k
                ], axis=1)
                raw_probs = pd_data['probe'].predict_proba(X)[:, 1]
                cal_probs, pi_te = rescale_probabilities_em(
                    raw_probs, pi_tr=pd_data['train_prevalence']
                )
                ntp_probs = syn_df_s['judgement_p_true'].to_numpy()

                print(f'  {s_label}: n={len(mids)}, pos={labels.sum()} '
                      f'({labels.mean():.1%}), pi_te={pi_te:.3f}')
                setting_results.append({
                    'label': s_label, 'color': color, 'ls': ls,
                    'probe_probs': cal_probs, 'ntp_probs': ntp_probs,
                    'labels': labels, 'is_syn': True,
                })

            else:  # real
                td       = test_data[test_ds]
                real_df  = td['real_df']
                syn_docs = set(pd_data['syn_document_ids'])
                mask     = ~real_df['document_id'].isin(syn_docs)
                idx      = np.where(mask.to_numpy())[0]
                mids     = real_df['measurement_id'].iloc[idx].tolist()
                labels   = td['real_labels'][idx]

                jdate    = JUDGE_DATES_REAL[test_ds][judge_model]
                real_act = load_activations(
                    test_ds, EXTRACTION_MODEL, EXTRACTION_DATES[test_ds], judge_model, jdate
                )
                X = np.concatenate([
                    np.stack([
                        np.array(real_act[str(mid)], dtype=np.float32)[l, h, :]
                        for mid in mids
                    ], axis=0)
                    for l, h in top_k
                ], axis=1)
                raw_probs = pd_data['probe'].predict_proba(X)[:, 1]
                cal_probs, pi_te = rescale_probabilities_em(
                    raw_probs, pi_tr=pd_data['train_prevalence']
                )
                ntp_probs = real_df[f'judgement_p_true_{judge_model}'].iloc[idx].to_numpy()

                n_gt = len(td['ground_truth_df'])
                ex_to_test_pos = {int(idx[i]): i for i in range(len(idx))}
                test_edges = [
                    (int(gt_i), ex_to_test_pos[int(ex_i)])
                    for (gt_i, ex_i), w in zip(td['edges'], td['edge_weights'])
                    if int(ex_i) in ex_to_test_pos and w > EDGE_THRESHOLD
                ]
                n_excl = int((~mask).sum())
                n_pap  = real_df.loc[~mask, 'document_id'].nunique()
                print(f'  {s_label}: n={len(idx)}, pos={labels.sum()} '
                      f'({labels.mean():.1%}), pi_te={pi_te:.3f}  '
                      f'[excl {n_excl} rows / {n_pap} syn papers]')
                setting_results.append({
                    'label': s_label, 'color': color, 'ls': ls,
                    'probe_probs': cal_probs, 'ntp_probs': ntp_probs,
                    'labels': labels, 'is_syn': False,
                    'test_edges': test_edges, 'n_gt': n_gt,
                })

    # ── Figure 1: Calibration curves ──────────────────────────────────────
    fig_cal, ax_cal = plt.subplots(figsize=(3.5, 3.5))
    ax_cal.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Perfect', zorder=1)

    for r in setting_results:
        # NTP baseline — faint dotted, same colour
        d_ntp = reliability_diagram_data(r['ntp_probs'], r['labels'])
        v_ntp = ~np.isnan(d_ntp['bin_accuracy'])
        ax_cal.plot(
            d_ntp['bin_centers'][v_ntp], d_ntp['bin_accuracy'][v_ntp],
            ':', color=r['color'], lw=0.8, alpha=0.30, zorder=2,
        )
        # Probe
        d_prb = reliability_diagram_data(r['probe_probs'], r['labels'])
        v_prb = ~np.isnan(d_prb['bin_accuracy'])
        ax_cal.plot(
            d_prb['bin_centers'][v_prb], d_prb['bin_accuracy'][v_prb],
            r['ls'], color=r['color'], lw=1.4, marker='o', ms=3.0,
            label=f"{r['label']} (ECE={d_prb['ece']:.3f})",
            zorder=3,
        )

    ax_cal.set_xlim(0, 1); ax_cal.set_ylim(0, 1)
    ax_cal.set_xlabel('Mean predicted probability')
    ax_cal.set_ylabel('Fraction positive')
    ax_cal.set_title(f'{judge_model} — trained on {train_dataset}', fontsize=8)
    ax_cal.legend(fontsize=6, loc='upper left', handlelength=2.5)
    ax_cal.grid(alpha=0.2)
    fig_cal.tight_layout()
    # fig_cal.savefig(
    #     FIGURES_DIR + f'cal_{judge_model}_{train_dataset}.pdf', bbox_inches='tight',
    # )
    plt.show()

    # ── Figure 2: Threshold sweep (2 × 2) ─────────────────────────────────
    fig_sw, axes = plt.subplots(2, 2, figsize=(6.5, 5.5),
                                sharex=True, sharey=True)
    axes_flat = axes.flatten()

    cmap = cm.viridis
    norm = mcolors.Normalize(vmin=THRESHOLD_SWEEP.min(), vmax=THRESHOLD_SWEEP.max())
    sm   = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    for ax, r in zip(axes_flat, setting_results):
        probe_rec, probe_hall = [], []
        ntp_rec,   ntp_hall   = [], []
        for t in THRESHOLD_SWEEP:
            if r['is_syn']:
                pr, ph = compute_rates_syn(r['probe_probs'], t, r['labels'])
                nr, nh = compute_rates_syn(r['ntp_probs'],   t, r['labels'])
            else:
                pr, ph = compute_rates(r['probe_probs'], t, r['labels'],
                                       r['test_edges'], r['n_gt'])
                nr, nh = compute_rates(r['ntp_probs'],   t, r['labels'],
                                       r['test_edges'], r['n_gt'])
            probe_rec.append(pr); probe_hall.append(ph)
            ntp_rec.append(nr);   ntp_hall.append(nh)

        pr_a = np.array(probe_rec);  ph_a = np.array(probe_hall)
        nr_a = np.array(ntp_rec);    nh_a = np.array(ntp_hall)
        v_pr = ~(np.isnan(pr_a) | np.isnan(ph_a))
        v_nr = ~(np.isnan(nr_a) | np.isnan(nh_a))

        # NTP — faint grey
        if v_nr.any():
            ax.plot(nh_a[v_nr], nr_a[v_nr], ':', color='#aaaaaa',
                    lw=0.9, alpha=0.5, zorder=2, label='NTP')
            ax.scatter(nh_a[v_nr], nr_a[v_nr], c=THRESHOLD_SWEEP[v_nr],
                       cmap=cmap, norm=norm, s=10, alpha=0.35, zorder=2)

        # Probe
        if v_pr.any():
            ax.plot(ph_a[v_pr], pr_a[v_pr], '-', color=r['color'],
                    lw=0.9, zorder=2, label='Probe')
            ax.scatter(ph_a[v_pr], pr_a[v_pr], c=THRESHOLD_SWEEP[v_pr],
                       cmap=cmap, norm=norm, s=25, zorder=3)

        # Mark threshold = 0.5
        idx0 = np.argmin(np.abs(THRESHOLD_SWEEP - 0.5))
        if v_pr[idx0]:
            ax.scatter([ph_a[idx0]], [pr_a[idx0]], s=55, c='none',
                       edgecolors='k', linewidths=1.0, zorder=4)

        ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
        ax.set_title(r['label'], fontsize=8)
        ax.legend(fontsize=6, loc='upper right')
        ax.grid(alpha=0.2)

    for ax in axes[1]:
        ax.set_xlabel('Hallucination rate')
    for ax in axes[:, 0]:
        ax.set_ylabel('Recovery rate')

    fig_sw.colorbar(sm, ax=axes[:, -1], label='Threshold',
                    fraction=0.046, pad=0.04)
    fig_sw.suptitle(
        f'Threshold sweep — {judge_model} trained on {train_dataset}',
        fontsize=9, y=1.01,
    )
    fig_sw.tight_layout()
    # fig_sw.savefig(
    #     FIGURES_DIR + f'sweep_{judge_model}_{train_dataset}.pdf', bbox_inches='tight',
    # )
    plt.show()

    # ── Summary table ─────────────────────────────────────────────────────
    rows = []
    for r in setting_results:
        for probs, kind in [(r['ntp_probs'], 'NTP'), (r['probe_probs'], 'Probe')]:
            if r['is_syn']:
                rec, hall = compute_rates_syn(probs, 0.5, r['labels'])
            else:
                rec, hall = compute_rates(probs, 0.5, r['labels'],
                                          r['test_edges'], r['n_gt'])
            m = _probe_metrics(probs, r['labels'])
            rows.append({
                'Test setting':  r['label'],
                'Type':          kind,
                'Recovery':      rec,
                'Hallucination': hall,
                'F1':            m['f1'],
                'AUROC':         m['auroc'],
                'ECE':           m['ece'],
            })

    df = pd.DataFrame(rows)
    print(f'\nSummary — {header}  (threshold = 0.5):')
    print(df.to_string(index=False, float_format='{:.3f}'.format))
    print()
    return df

## Run All Combinations

In [ ]:
all_results = {}
for jm in JUDGE_MODELS:
    for train_ds in DATASETS:
        all_results[(jm, train_ds)] = analyze_probe(jm, train_ds)